In [2]:
import torchvision.datasets as datasets
kmnist_train = datasets.KMNIST(root='./data', train=True, download=True)
kmnist_test = datasets.KMNIST(root='./data', train=False, download=True)

100.0%
100.0%
100.0%
100.0%


In [ ]:
import torch
import torchvision
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import os


data_root = './data'
save_path = './PAC/processed_kmnist.pt'
n_components = 4
classes_to_keep = [0, 1, 2, 3] 
os.makedirs(os.path.dirname(save_path), exist_ok=True)

print("1. Loading KMNIST data...")
train_set = torchvision.datasets.KMNIST(root=data_root, train=True, download=True)
test_set = torchvision.datasets.KMNIST(root=data_root, train=False, download=True)
X_train = train_set.data.numpy().reshape(-1, 784)
y_train = train_set.targets.numpy()

X_test = test_set.data.numpy().reshape(-1, 784)
y_test = test_set.targets.numpy()

print(f"   Original data shape: {X_train.shape}")

print(f"2. Filtering classes {classes_to_keep}...")
def filter_and_remap(X, y, classes):
    # Create mask to keep only specified classes
    mask = np.isin(y, classes)
    X_filtered = X[mask]
    y_filtered = y[mask]
    
    # Remap labels: 0->0, 1->1, 2->2, 3->3 (to avoid later errors)
    map_dict = {c: i for i, c in enumerate(classes)}
    y_mapped = np.array([map_dict[val] for val in y_filtered])
    
    return X_filtered, y_mapped

X_train, y_train = filter_and_remap(X_train, y_train, classes_to_keep)
X_test, y_test = filter_and_remap(X_test, y_test, classes_to_keep)

print(f"   Filtered shape: {X_train.shape}")

print(f"3. PCA dimensionality reduction (784 -> {n_components})...")
pca = PCA(n_components=n_components)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test) # Test set only transform

print("4. Normalizing to [0, pi] (suitable for quantum encoding)...")
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("5. Saving processed data...")
processed_data = {
    "X_train": torch.tensor(X_train, dtype=torch.float32),
    "y_train": torch.tensor(y_train, dtype=torch.long),
    "X_test": torch.tensor(X_test, dtype=torch.float32),
    "y_test": torch.tensor(y_test, dtype=torch.long)
}

torch.save(processed_data, save_path)
print(f"Done! File saved to: {save_path}")
